# Lecture 1 author-facing figure notebook: task examples

This notebook generates the figures used in the Lecture 1 sections
"From inference to machine-learning problems" and "What do we use learned
models for?" in `slide_deck.typ`. It follows the same course figure
convention as `coin_model.ipynb`:

- the notebook lives in the lecture directory;
- generated assets go in `figures/`;
- each filename is `<notebook-stem>__<figure-name>.<ext>`;
- the figure path is shown on the slide as provenance.

## The two jobs this notebook does

Section 3 of the lecture makes two points at once, so the figures come in two
families.

**Family A — a different canonical dataset for each task.** Prediction,
generation, discovery, and action are illustrated in four visibly different
problem domains, so that students see the breadth of probabilistic machine
learning. These are adaptations of figures from Kevin Murphy,
*Probabilistic Machine Learning: Advanced Topics*.

**Family B — one dataset, four questions.** The *same* two Iris petal
measurements are used to pose all four questions, so that students see that a
task is not a property of a dataset. These are newly constructed for this
course; they are not adaptations of a Murphy figure.

Family B exists to block a misreading that Family A can invite: that
prediction "is" a classification dataset, generation "is" images, and
discovery "is" a scatter of unlabeled points. The panels are deliberately
drawn in the same axes, with the same 100 flowers, so that the only thing
that changes between panels is the question.

## Provenance

### Family A — different datasets for different tasks

| Lecture role | Dataset | Task | Murphy figure / chapter | Source notebook | Status | Modelling assumptions |
|---|---|---|---|---|---|---|
| Canonical prediction example | Iris, petal length/width, versicolor vs. virginica (`sklearn`) | Prediction | Fig. 15.8, Ch. 15 *Generalized Linear Models* | `logreg_iris_bayes_2d.ipynb` | **Reproduced**, with one deliberate change: axes are in centimetres rather than standardized units, so that this figure and the Family B panels show the flowers in the same coordinates. The model is fit on standardized features exactly as before. | Logistic likelihood; zero-mean Gaussian prior on the weights with precision 1; Laplace approximation to the posterior (MAP plus inverse Hessian); posterior predictive approximated by averaging over 40 posterior weight samples. |
| Canonical generation example | Olivetti faces (`sklearn`), 400 images at 64x64 grayscale | Generation | Fig. 21.2, Ch. 21 *Variational Autoencoders* | `celeba_vae_ae_comparison.ipynb` | **Adapted, not reproduced at scale.** The original trains a convolutional VAE on CelebA (~200k 64x64 RGB face crops) on a GPU. For a fast, dependency-light, CPU-only lecture build we substitute Olivetti faces and a small fully-connected VAE trained for about ten seconds. The conceptual structure is identical; photorealism is sacrificed. The blur is a genuine property of pixel-wise VAE likelihoods, not an artefact of the substitution, and the slide's speaker note says so. | 16-dimensional standard normal latent; one hidden layer of 256 units in encoder and decoder; Bernoulli (binary cross-entropy) pixel likelihood; amortized Gaussian variational posterior; ELBO trained with Adam. |
| Canonical discovery example | Synthetic 2D data, three Gaussian components with distinct covariance | Discovery | Fig. 28.2, Ch. 28 *Latent Factor Models* | `gmm_2d.ipynb` | **Reproduced** structurally: unlabeled data on the left, inferred latent grouping with covariance ellipses on the right. | Three-component Gaussian mixture, full covariance, fit by EM (`sklearn.mixture.GaussianMixture`, `n_init=5`). |
| Canonical action example | Synthetic 1D regression, `f(x) = sin(10x) * x` on `[0, 1]` | Action | Fig. 34.16, Ch. 34.7 *Active Learning* | `bayes_opt_vs_active_learning.ipynb` | **Reproduced, active-learning half only.** Murphy's figure contrasts active learning with Bayesian optimization; only the active-learning panel is reproduced, because the Bayesian-optimization comparison is a different point and would make the slide too busy. The original's JAX/`tinygp`/`jaxopt` stack is replaced by an exact GP written in NumPy/SciPy, to avoid adding heavy dependencies for one figure. Same target function, same noise level, same two initial points and six greedy acquisitions, same maximum-variance acquisition rule. One addition: the *next* query is marked explicitly, since "the model chooses where to measure next" is the lecture's point. | Zero-mean GP, squared-exponential kernel; signal variance and length scale fit by maximizing the log marginal likelihood; observation noise fixed at 0.05 as in the original; acquisition is posterior standard deviation, which for a Gaussian is a monotone function of predictive entropy. |

### Family B — same dataset, different questions

All four panels use the identical data matrix: the 100 Iris flowers with
`target != 0` (versicolor and virginica), with petal length and petal width in
centimetres. Nothing else is shared — each panel fits its own model and asks
its own question. None of these is a Murphy figure; they are newly constructed
for this lecture.

| Lecture role | Observed | Unknown / target | Question | Object | Status | Modelling assumptions |
|---|---|---|---|---|---|---|
| Prediction panel | measurements `x` | species `y` | Given this flower's measurements, which species is it? | `p(y \| x, D)` | Newly constructed; same model as the Family A prediction figure. | As in Family A prediction, drawn compactly and without an in-figure title. |
| Generation panel | species `y` | new measurements `x` | What would another plausible flower of this species look like? | `x ~ p(x \| y, D)` | Newly constructed. | One maximum-likelihood bivariate Gaussian per species, `p(x \| y) = N(mu_y, Sigma_y)`, with 40 samples drawn per species. The Gaussian is a *teaching* choice, not a claim about flower morphology; it is the simplest class-conditional density that makes "class in, measurement out" visible. |
| Discovery panel | measurements `x` | cluster `z` | If nobody told us the species, what structure is in the measurements? | `p(z \| x, D)` from `p(z) p(x \| z)` | Newly constructed. | Two-component Gaussian mixture, full covariance, fit by EM on `x` alone. **The species labels are never passed to the fit.** They are used only after the fact, in the speaker note, to report how far the discovered grouping happens to agree with the species. |
| Action panel | a small labeled subset plus an unlabeled pool | which flower to label next | Which flower should we pay to label next? | `x_next = argmax_x H[p(y \| x, D)]` | Newly constructed. | 10 labeled flowers, 5 drawn at random per species; the remaining 90 form the pool. The same Bayesian logistic regression is fit to the labeled subset only. The acquisition is the predictive entropy of the posterior-averaged class probability; uncertainty sampling picks its maximum over the pool. |
| Combined 2x2 | all four | | | | Newly constructed. | A convenience composite of the four panels *with* in-figure titles, for handouts and for reference. The slide deck does **not** use it: it composes the four separate panel files in a Typst grid, so panels can be placed and revealed independently. |

## Reproducibility

Every figure draws from its own `numpy` generator, seeded from `BASE_SEED`
plus a per-figure offset (see `FIGURE_SEED` below). This means cells can be
added, removed, re-ordered, or re-run individually without changing any other
figure — which matters here, because the VAE cell is the one cell that needs
`torch` and a reader may want to skip it.

In [1]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Every figure draws from its own independent stream, so that adding,
# removing, or re-running a cell cannot change any other figure.
BASE_SEED = 20260915
FIGURE_SEED = {
    "prediction_iris_logistic": 1,
    "generation_vae_samples": 2,
    "discovery_gmm_clusters": 3,
    "action_active_learning": 4,
    "iris_prediction": 11,
    "iris_generation": 12,
    "iris_discovery": 13,
    "iris_action": 14,
}

def make_rng(figure_name):
    'A dedicated generator for one figure, so cells stay order-independent.'
    return np.random.default_rng(BASE_SEED + FIGURE_SEED[figure_name])

# Works both locally and in Colab.
OUT = Path("figures")
OUT.mkdir(exist_ok=True)

PREFIX = "task_examples_figures__"

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

def save_fig(fig, name, **kwargs):
    path = OUT / f"{PREFIX}{name}"
    fig.savefig(path, bbox_inches="tight", **kwargs)
    print(f"saved {path}")
    return path

## Shared building blocks: the Iris subset and a Bayesian logistic regression

Both families need the same two things, so they are defined once here rather
than twice.

The dataset is the standard two-class Iris subset used in Murphy Fig. 15.8:
the 100 *versicolor* and *virginica* flowers, described by petal length and
petal width. Plotting in centimetres rather than standardized units costs
nothing and makes the "same flowers, four questions" argument easier to see.

The model is a Bayesian logistic regression fit by MAP estimation with a
Laplace approximation to the posterior: a Gaussian centred at the MAP estimate
with covariance given by the inverse Hessian of the negative log posterior.
The predictive object is the full distribution, approximated by averaging over
posterior weight samples, not a single point estimate.

In [2]:
from sklearn.datasets import load_iris

iris = load_iris()
_X_all, _y_all = iris.data, iris.target

# Matches Murphy Fig. 15.8: petal length and petal width, versicolor vs virginica.
_two_class = _y_all != 0
X_IRIS = _X_all[_two_class][:, [2, 3]]          # centimetres
Y_IRIS = (_y_all[_two_class] - 1).astype(float)  # 0 = versicolor, 1 = virginica

SPECIES_COLOR = {0: "C0", 1: "C3"}
SPECIES_NAME = {0: "versicolor", 1: "virginica"}
CLUSTER_COLOR = ("C4", "C2")  # deliberately not the species colours

XLABEL, YLABEL = "petal length (cm)", "petal width (cm)"
XLIM, YLIM = (2.6, 7.4), (0.85, 2.75)

print(f"{X_IRIS.shape[0]} flowers, {int((Y_IRIS == 0).sum())} versicolor, "
      f"{int((Y_IRIS == 1).sum())} virginica")

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
from scipy.optimize import minimize
from matplotlib.patches import Ellipse


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def fit_bayes_logreg(X, y, prior_precision=1.0):
    'MAP fit plus a Laplace approximation, on standardized features.'
    mu, sd = X.mean(0), X.std(0)
    Phi = np.column_stack([np.ones(len(X)), (X - mu) / sd])

    def neg_log_posterior(w):
        z = Phi @ w
        log_lik = np.sum(y * z - np.logaddexp(0.0, z))
        log_prior = -0.5 * prior_precision * np.sum(w ** 2)
        return -(log_lik + log_prior)

    def grad_neg_log_posterior(w):
        p = sigmoid(Phi @ w)
        return -(Phi.T @ (y - p) - prior_precision * w)

    w_map = minimize(neg_log_posterior, np.zeros(Phi.shape[1]),
                     jac=grad_neg_log_posterior, method="BFGS").x

    p_map = sigmoid(Phi @ w_map)
    hessian = Phi.T @ np.diag(p_map * (1 - p_map)) @ Phi + prior_precision * np.eye(Phi.shape[1])
    return {"w_map": w_map, "cov": np.linalg.inv(hessian), "mu": mu, "sd": sd}


def design_matrix(fit, X):
    return np.column_stack([np.ones(len(X)), (X - fit["mu"]) / fit["sd"]])


def posterior_weight_samples(fit, rng, n_samples=200):
    return rng.multivariate_normal(fit["w_map"], fit["cov"], size=n_samples)


def posterior_class_probs(fit, X, rng, n_samples=200):
    'Columns are posterior samples of p(y=1 | x, w); average them for p(y | x, D).'
    return sigmoid(design_matrix(fit, X) @ posterior_weight_samples(fit, rng, n_samples).T)


def binary_entropy(p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))


def iris_grid(n=160):
    g1, g2 = np.linspace(*XLIM, n), np.linspace(*YLIM, n)
    G1, G2 = np.meshgrid(g1, g2)
    return G1, G2, np.column_stack([G1.ravel(), G2.ravel()])


def covariance_ellipse(ax, mean, cov, color, n_std=2.0, **kwargs):
    values, vectors = np.linalg.eigh(cov)
    order = values.argsort()[::-1]
    values, vectors = values[order], vectors[:, order]
    angle = np.degrees(np.arctan2(*vectors[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(values)
    ax.add_patch(Ellipse(mean, width, height, angle=angle,
                         facecolor="none", edgecolor=color, **kwargs))


# A white, borderless legend box: readable on top of a filled contour plot
# without drawing a competing rectangle.
LEGEND_STYLE = dict(frameon=True, facecolor="white", framealpha=0.85, edgecolor="none")

---

# Family A — a different canonical dataset for each task

Four tasks, four visibly different problem domains. The point of this family
is breadth: prediction, generation, discovery, and action are not variations
on one picture.

## A1. Prediction — Bayesian logistic regression on Iris

Given measured flower features $x_*$, what should we say about the class
$y_*$? The predictive object is the full distribution

$$p(y_* \mid x_*, D) = \int p(y_* \mid x_*, \theta)\, p(\theta \mid D)\, d\theta,$$

approximated here by averaging over posterior samples of $\theta$. The fan of
sampled decision boundaries *is* the posterior uncertainty; the heavy line is
the boundary of the posterior-averaged predictive probability.

In [ ]:
rng = make_rng("prediction_iris_logistic")

fit = fit_bayes_logreg(X_IRIS, Y_IRIS)
print("MAP weights:", fit["w_map"])
print("Posterior standard deviations:", np.sqrt(np.diag(fit["cov"])))

G1, G2, grid = iris_grid()
n_boundaries = 40
probs = posterior_class_probs(fit, grid, rng, n_samples=n_boundaries)
prob_mean = probs.mean(axis=1).reshape(G1.shape)

fig, ax = plt.subplots(figsize=(7, 5.5))
contour = ax.contourf(G1, G2, prob_mean, levels=20, cmap="RdBu_r",
                      alpha=0.75, vmin=0, vmax=1)
for s in range(n_boundaries):
    ax.contour(G1, G2, probs[:, s].reshape(G1.shape), levels=[0.5],
               colors="k", linewidths=0.6, alpha=0.3)
ax.contour(G1, G2, prob_mean, levels=[0.5], colors="k", linewidths=2.2)
for label in (0, 1):
    ax.scatter(*X_IRIS[Y_IRIS == label].T, c=SPECIES_COLOR[label], edgecolor="k",
               s=45, lw=0.6, label=SPECIES_NAME[label])
ax.set(xlabel=XLABEL, ylabel=YLABEL, xlim=XLIM, ylim=YLIM,
       title="Bayesian logistic regression: posterior decision boundaries")
ax.legend(loc="upper left", **LEGEND_STYLE)
fig.colorbar(contour, ax=ax, label=r"$p(y=\mathrm{virginica}\mid x)$")
fig.tight_layout()

save_fig(fig, "prediction_iris_logistic.svg")
plt.show()

## A2. Generation — sampling new faces from a learned latent-variable model

A generative task asks the model to produce new, plausible objects, not just a
summary of an existing one. We train a small variational autoencoder,
$z \sim p(z) = \mathcal{N}(0, I)$, $x \mid z \sim p_\theta(x \mid z)$, by
maximizing an evidence lower bound. New faces are produced by sampling
$z^{(s)} \sim p(z)$ and decoding $x^{(s)} \sim p_\theta(x \mid z^{(s)})$.

See the provenance table: this is a lightweight CPU adaptation of Murphy's
CelebA VAE figure using the much smaller Olivetti faces dataset, so treat the
samples as illustrating the *mechanism*, not as a photorealism benchmark.

**This is the only cell in the notebook that needs `torch`.** Every other
figure uses its own generator (see `make_rng`), so skipping this cell changes
nothing else.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import fetch_olivetti_faces

torch.manual_seed(BASE_SEED + FIGURE_SEED["generation_vae_samples"])

faces = fetch_olivetti_faces(shuffle=True, random_state=BASE_SEED)
X_faces = faces.images.astype(np.float32)  # (400, 64, 64), already in [0, 1]
n_faces, H, W = X_faces.shape
X_faces_t = torch.from_numpy(X_faces.reshape(n_faces, -1))

D, LATENT_DIM, HIDDEN_DIM = H * W, 16, 256


class SmallVAE(nn.Module):
    'A deliberately small, fully-connected VAE for fast CPU training.'

    def __init__(self):
        super().__init__()
        self.enc1 = nn.Linear(D, HIDDEN_DIM)
        self.enc_mu = nn.Linear(HIDDEN_DIM, LATENT_DIM)
        self.enc_logvar = nn.Linear(HIDDEN_DIM, LATENT_DIM)
        self.dec1 = nn.Linear(LATENT_DIM, HIDDEN_DIM)
        self.dec2 = nn.Linear(HIDDEN_DIM, D)

    def encode(self, x):
        h = F.relu(self.enc1(x))
        return self.enc_mu(h), self.enc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def decode(self, z):
        h = F.relu(self.dec1(z))
        return torch.sigmoid(self.dec2(h))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def vae_loss(recon, x, mu, logvar):
    reconstruction = F.binary_cross_entropy(recon, x, reduction="sum")
    kl_divergence = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return reconstruction + kl_divergence


model = SmallVAE()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_epochs, batch_size = 150, 64
for epoch in range(n_epochs):
    perm = torch.randperm(n_faces)
    for i in range(0, n_faces, batch_size):
        batch = X_faces_t[perm[i : i + batch_size]]
        optimizer.zero_grad()
        recon, mu, logvar = model(batch)
        loss = vae_loss(recon, batch, mu, logvar)
        loss.backward()
        optimizer.step()

print("training complete")

In [ ]:
rng = make_rng("generation_vae_samples")

model.eval()
with torch.no_grad():
    real_idx = rng.choice(n_faces, size=8, replace=False)
    real_imgs = X_faces_t[real_idx].numpy().reshape(-1, H, W)
    z_samples = torch.randn(8, LATENT_DIM)
    gen_imgs = model.decode(z_samples).numpy().reshape(-1, H, W)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for i in range(8):
    axes[0, i].imshow(real_imgs[i], cmap="gray", vmin=0, vmax=1)
    axes[1, i].imshow(gen_imgs[i], cmap="gray", vmin=0, vmax=1)
    for row in (0, 1):
        axes[row, i].set_xticks([])
        axes[row, i].set_yticks([])
        for spine in axes[row, i].spines.values():
            spine.set_visible(False)

axes[0, 0].set_ylabel("training\ndata", rotation=0, ha="right", va="center")
axes[1, 0].set_ylabel(r"$x^{(s)}\!\sim p_\theta(x\mid z^{(s)})$", rotation=0,
                      ha="right", va="center")
fig.suptitle("Generation: sampling new faces from a learned latent-variable model", y=1.02)
fig.tight_layout()

save_fig(fig, "generation_vae_samples.png", dpi=150)
plt.show()

## A3. Discovery — inferring latent structure with a Gaussian mixture

Here we are not predicting a supplied label, nor synthesizing a new
observation. We are inferring hidden structure — which group each point
belongs to — that organizes unlabeled observations: $z \sim p(z)$,
$x \mid z \sim p(x \mid z)$, and after observing $x$ we compute the posterior
$p(z \mid x)$ over cluster membership.

A Gaussian mixture is, formally, a generative model of $x$. The teaching point
here (as in Murphy Ch. 28) is that the *task* it is being used for is
discovery: recovering meaningful latent variables, not sampling new
realistic-looking data.

In [ ]:
from sklearn.mixture import GaussianMixture

rng = make_rng("discovery_gmm_clusters")

# Three elliptical clusters with distinct covariance, presented unlabeled.
means = [(-2.5, 1.0), (2.0, 2.0), (0.0, -2.5)]
covariances = [
    [[0.6, 0.35], [0.35, 0.4]],
    [[0.5, -0.3], [-0.3, 0.9]],
    [[1.0, 0.0], [0.0, 0.25]],
]
sizes = [110, 90, 130]

X_clusters = np.vstack([rng.multivariate_normal(mean, cov, size=size)
                        for mean, cov, size in zip(means, covariances, sizes)])
X_clusters = X_clusters[rng.permutation(len(X_clusters))]

gmm = GaussianMixture(n_components=3, covariance_type="full",
                      random_state=0, n_init=5)
z_hat = gmm.fit_predict(X_clusters)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))

axes[0].scatter(X_clusters[:, 0], X_clusters[:, 1], s=22, c="0.35", alpha=0.75)
axes[0].set(title="Observed data (unlabeled)", xlabel=r"$x_1$", ylabel=r"$x_2$")

colors = ["C0", "C1", "C2"]
order = np.argsort(gmm.means_[:, 0])
color_for = {component: colors[i] for i, component in enumerate(order)}
axes[1].scatter(X_clusters[:, 0], X_clusters[:, 1], s=22,
                c=[color_for[k] for k in z_hat], alpha=0.8)
for k in range(3):
    covariance_ellipse(axes[1], gmm.means_[k], gmm.covariances_[k],
                       color_for[k], linewidth=2.2)
    axes[1].scatter(*gmm.means_[k], marker="x", s=90, c="k", linewidths=2.2)
axes[1].set(title=r"Inferred latent structure: $p(z \mid x)$",
            xlabel=r"$x_1$", ylabel=r"$x_2$")

for ax in axes:
    ax.set_xlim(-5.5, 5)
    ax.set_ylim(-4.5, 4.5)
    ax.set_aspect("equal")

fig.tight_layout()
save_fig(fig, "discovery_gmm_clusters.svg")
plt.show()

## A4. Action — active learning with a Gaussian process

This is the fourth task, and the one that lies outside the syllabus (Murphy
Part VI, Ch. 34). It gets a real computational figure anyway, because "action"
should not arrive as an abstract equation when the other three tasks arrive as
pictures.

Reproducing the active-learning half of Murphy Fig. 34.16: an unknown function
$f(x) = \sin(10x)\,x$ is measured with noise; a GP is fit to the measurements
so far; and the next measurement location is chosen by maximizing an
acquisition function — here the posterior standard deviation, so the algorithm
measures where it is currently least certain.

$$x_{\text{next}} = \arg\max_x\ \alpha(x;\, D), \qquad
\alpha(x) = \sqrt{\mathbb{V}[f(x) \mid D]}.$$

The model does not merely *report* uncertainty. It *acts on* it.

The original notebook uses JAX with `tinygp` and `jaxopt`. We use an exact GP
written in NumPy/SciPy instead, to avoid adding a heavy dependency for one
figure; the target function, noise level, initial points, acquisition rule and
number of acquisitions are unchanged.

In [ ]:
from scipy.linalg import cho_solve, cholesky

NOISE_STD = 0.05  # as in the original notebook


def true_f(x):
    return np.sin(10 * x) * x


def _kernel(a, b, variance, length_scale):
    'Squared-exponential kernel.'
    return variance * np.exp(-0.5 * (a[:, None] - b[None, :]) ** 2 / length_scale ** 2)


def _neg_log_marginal_likelihood(log_params, x, y):
    variance, length_scale = np.exp(log_params)
    K = _kernel(x, x, variance, length_scale) + (NOISE_STD ** 2 + 1e-8) * np.eye(len(x))
    try:
        L = cholesky(K, lower=True)
    except np.linalg.LinAlgError:
        return 1e10
    alpha = cho_solve((L, True), y)
    return float(0.5 * y @ alpha + np.log(np.diag(L)).sum()
                 + 0.5 * len(x) * np.log(2 * np.pi))


def gp_fit(x, y):
    'Signal variance and length scale by maximum marginal likelihood.'
    solution = minimize(_neg_log_marginal_likelihood, np.log([1.0, 1.0]),
                        args=(x, y), method="Nelder-Mead",
                        options=dict(xatol=1e-6, fatol=1e-8, maxiter=2000))
    return np.exp(solution.x)


def gp_predict(x, y, x_star, variance, length_scale):
    K = _kernel(x, x, variance, length_scale) + (NOISE_STD ** 2 + 1e-8) * np.eye(len(x))
    L = cholesky(K, lower=True)
    alpha = cho_solve((L, True), y)
    K_star = _kernel(x, x_star, variance, length_scale)
    mean = K_star.T @ alpha
    var = variance - np.sum(K_star * cho_solve((L, True), K_star), axis=0)
    return mean, np.maximum(var, 1e-12)

In [ ]:
rng = make_rng("action_active_learning")

x_grid = np.linspace(0, 1, 100)
y_grid = true_f(x_grid) + rng.normal(0, NOISE_STD, size=x_grid.shape)

# Two initial measurements; every other grid point is an available query.
initial = np.array([12, 73])
x_train, y_train = x_grid[initial].copy(), y_grid[initial].copy()
pool = np.setdiff1d(np.arange(len(x_grid)), initial)

N_ACQUISITIONS = 6  # as in the original figure
acquired = []
for _ in range(N_ACQUISITIONS):
    variance, length_scale = gp_fit(x_train, y_train)
    _, pool_var = gp_predict(x_train, y_train, x_grid[pool], variance, length_scale)
    best = int(np.argmax(pool_var))
    acquired.append(pool[best])
    x_train = np.append(x_train, x_grid[pool[best]])
    y_train = np.append(y_train, y_grid[pool[best]])
    pool = np.delete(pool, best)

variance, length_scale = gp_fit(x_train, y_train)
mean, var = gp_predict(x_train, y_train, x_grid, variance, length_scale)
sd = np.sqrt(var)

# The next query the algorithm would make, given everything measured so far.
_, pool_var = gp_predict(x_train, y_train, x_grid[pool], variance, length_scale)
next_query = pool[int(np.argmax(pool_var))]

print(f"signal variance {variance:.3f}, length scale {length_scale:.3f}")
print(f"next query at x = {x_grid[next_query]:.3f}")

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(7.4, 5.4), sharex=True,
                       gridspec_kw=dict(height_ratios=[2, 1], hspace=0.16))

ax[0].fill_between(x_grid, mean - 2 * sd, mean + 2 * sd, color="C0", alpha=0.28,
                   label="95% credible band")
ax[0].plot(x_grid, true_f(x_grid), color="0.35", lw=1.6, label="true function $f(x)$")
ax[0].plot(x_grid, mean, color="C2", ls="--", lw=1.8, label="predictive mean")
ax[0].scatter(x_grid[initial], y_grid[initial], c="k", s=46, zorder=10,
              label="initial measurements")
ax[0].scatter(x_grid[acquired], y_grid[acquired], c="C3", edgecolor="k", lw=0.6,
              s=52, zorder=9, label="actively chosen measurements")
ax[0].axvline(x_grid[next_query], color="k", ls=":", lw=1.5)
ax[0].set(ylim=(-1.25, 1.2), ylabel="$y$",
          title="Active learning: the model chooses where to measure next")
ax[0].legend(loc="lower left", fontsize=8.5, ncol=2, **LEGEND_STYLE)

ax[1].plot(x_grid, sd, color="C4", lw=1.8,
           label=r"acquisition $\alpha(x)=\sqrt{\mathbb{V}[f(x)\mid\mathcal{D}]}$")
ax[1].fill_between(x_grid, 0, sd, color="C4", alpha=0.18)
ax[1].axvline(x_grid[next_query], color="k", ls=":", lw=1.5)
ax[1].scatter([x_grid[next_query]], [sd[next_query]], s=90, facecolor="none",
              edgecolor="k", lw=2.2, zorder=6)
ax[1].annotate("measure here next", xy=(x_grid[next_query], sd[next_query]),
               xytext=(x_grid[next_query] + 0.05, sd.max() * 1.20),
               fontsize=10, fontweight="bold",
               bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.3", lw=0.8),
               arrowprops=dict(arrowstyle="->", lw=1.5, color="k"))
ax[1].legend(loc="upper right", fontsize=9.5, **LEGEND_STYLE)
ax[1].set(xlabel="$x$", ylabel=r"$\alpha(x)$", ylim=(0, sd.max() * 1.40))

save_fig(fig, "action_active_learning.svg")
plt.show()

---

# Family B — one dataset, four questions

Family A can leave a wrong impression: that prediction *is* a labeled
classification dataset, generation *is* images, and discovery *is* a scatter
of unlabeled points. It is not so. Prediction, generation, discovery, and
action are questions we choose to ask, not properties a dataset carries.

So here we ask all four of them of the same 100 flowers, described by the same
two measurements, drawn in the same axes.

| | Observed | Unknown | Question | Object |
|---|---|---|---|---|
| Prediction | measurements $x$ | species $y$ | Which species is this flower? | $p(y \mid x, D)$ |
| Generation | species $y$ | measurements $x$ | What would another flower of this species look like? | $x \sim p(x \mid y, D)$ |
| Discovery | measurements $x$ | cluster $z$ | What grouping is in the measurements? | $p(z \mid x, D)$ |
| Action | labeled subset + pool | which flower to label | Which flower should we label next? | $x_{\text{next}} = \arg\max_x \alpha(x; D)$ |

These conditional forms are *not* definitions of the four task categories.
Prediction is not "the task whose object is $p(y \mid x)$" — a generative
classifier predicts through $p(x \mid y)p(y)$, and a conditional generator has
the form $p(y \mid x)$. The table says what each of *these four panels* does,
which is exactly the point: the variables did not change, the question did.

Each panel is saved as its own file, without an in-figure title, so the deck
can place, resize, and reveal them independently and set the question text in
the slide's own typography. A combined 2x2 version with titles is also saved,
for handouts and reference.

In [ ]:
from sklearn.mixture import GaussianMixture
from matplotlib.lines import Line2D

PANEL_TITLES = {
    "prediction": ("Prediction: which species is this flower?",
                   r"$p(y \mid x,\ \mathcal{D})$"),
    "generation": ("Generation: what would another flower look like?",
                   r"$x \sim p(x \mid y,\ \mathcal{D})$"),
    "discovery": ("Discovery: what grouping is in the measurements?",
                  r"$p(z \mid x,\ \mathcal{D})$"),
    "action": ("Action: which flower should we label next?",
               r"$x_{\mathrm{next}} = \arg\max_x\ \mathbb{H}[p(y \mid x,\ \mathcal{D})]$"),
}


def frame_panel(ax, key, titled):
    ax.set(xlim=XLIM, ylim=YLIM, xlabel=XLABEL, ylabel=YLABEL)
    if titled:
        question, math = PANEL_TITLES[key]
        ax.set_title(f"{question}\n{math}", fontsize=11.5, linespacing=1.5)


def panel_prediction(ax, rng, titled=False, n_boundaries=40):
    'Observed x, unknown y. Measurements in, class out.'
    fit = fit_bayes_logreg(X_IRIS, Y_IRIS)
    G1, G2, grid = iris_grid()
    probs = posterior_class_probs(fit, grid, rng, n_samples=n_boundaries)
    prob_mean = probs.mean(axis=1).reshape(G1.shape)

    ax.contourf(G1, G2, prob_mean, levels=20, cmap="RdBu_r", alpha=0.7, vmin=0, vmax=1)
    for s in range(n_boundaries):
        ax.contour(G1, G2, probs[:, s].reshape(G1.shape), levels=[0.5],
                   colors="k", linewidths=0.6, alpha=0.3)
    ax.contour(G1, G2, prob_mean, levels=[0.5], colors="k", linewidths=2.2)
    for label in (0, 1):
        ax.scatter(*X_IRIS[Y_IRIS == label].T, c=SPECIES_COLOR[label],
                   edgecolor="k", s=38, lw=0.6, label=SPECIES_NAME[label])
    ax.legend(loc="upper left", fontsize="small", **LEGEND_STYLE)
    frame_panel(ax, "prediction", titled)


def panel_generation(ax, rng, titled=False, n_new=40):
    'Condition on y, generate x. Class in, plausible measurement vector out.'
    ax.scatter(*X_IRIS.T, facecolor="none", edgecolor="0.6", s=34, lw=0.9)
    handles = [Line2D([], [], marker="o", ls="", mfc="none", mec="0.6", ms=6,
                      label="observed flowers")]
    for label in (0, 1):
        X_k = X_IRIS[Y_IRIS == label]
        mean_k, cov_k = X_k.mean(0), np.cov(X_k.T)
        covariance_ellipse(ax, mean_k, cov_k, SPECIES_COLOR[label],
                           linewidth=1.8, linestyle="--", alpha=0.9)
        X_new = rng.multivariate_normal(mean_k, cov_k, size=n_new)
        ax.scatter(*X_new.T, marker="D", c=SPECIES_COLOR[label], edgecolor="k",
                   s=34, lw=0.5, alpha=0.95)
        handles.append(Line2D([], [], marker="D", ls="", mfc=SPECIES_COLOR[label],
                              mec="k", ms=6,
                              label=f"generated {SPECIES_NAME[label]}"))
    ax.legend(handles=handles, loc="upper left", fontsize="x-small", **LEGEND_STYLE)
    frame_panel(ax, "generation", titled)


def panel_discovery(ax, rng, titled=False):
    'Observed x, latent z. The species labels are never passed to the fit.'
    mixture = GaussianMixture(n_components=2, covariance_type="full",
                              random_state=0, n_init=10).fit(X_IRIS)
    z_hat = mixture.predict(X_IRIS)

    order = list(np.argsort(mixture.means_[:, 0]))
    for component in np.unique(z_hat):
        color = CLUSTER_COLOR[order.index(component)]
        ax.scatter(*X_IRIS[z_hat == component].T, c=color, edgecolor="k", s=38,
                   lw=0.6, label=f"cluster $z={order.index(component) + 1}$")
        covariance_ellipse(ax, mixture.means_[component],
                           mixture.covariances_[component], color, linewidth=2.0)
        ax.scatter(*mixture.means_[component], marker="x", c="k", s=70, lw=2)
    ax.legend(loc="upper left", fontsize="small", title="labels never used",
              title_fontsize="x-small", **LEGEND_STYLE)
    frame_panel(ax, "discovery", titled)

    # Reported only after the fact, for the speaker note. Not used to fit.
    return max((z_hat == Y_IRIS).mean(), (z_hat == 1 - Y_IRIS).mean())


def panel_action(ax, rng, titled=False, n_labeled=10):
    'Labels are expensive. Use the model to choose which flower to label next.'
    indices = np.arange(len(X_IRIS))
    labeled = np.concatenate([
        rng.choice(indices[Y_IRIS == label], size=n_labeled // 2, replace=False)
        for label in (0, 1)
    ])
    pool = np.setdiff1d(indices, labeled)

    fit = fit_bayes_logreg(X_IRIS[labeled], Y_IRIS[labeled])

    G1, G2, grid = iris_grid()
    grid_prob = posterior_class_probs(fit, grid, rng).mean(axis=1)
    ax.contourf(G1, G2, binary_entropy(grid_prob).reshape(G1.shape),
                levels=np.linspace(0, np.log(2), 16), cmap="Purples",
                alpha=0.32, vmin=0, vmax=np.log(2))
    ax.contour(G1, G2, grid_prob.reshape(G1.shape), levels=[0.5],
               colors="k", linewidths=1.3, linestyles="--")

    pool_entropy = binary_entropy(posterior_class_probs(fit, X_IRIS[pool], rng).mean(axis=1))
    next_query = pool[int(np.argmax(pool_entropy))]

    ax.scatter(*X_IRIS[pool].T, facecolor="none", edgecolor="0.3", lw=1.0,
               s=14 + 120 * (pool_entropy / np.log(2)) ** 2,
               label="unlabeled pool (size = uncertainty)")
    for label in (0, 1):
        selected = labeled[Y_IRIS[labeled] == label]
        ax.scatter(*X_IRIS[selected].T, c=SPECIES_COLOR[label], edgecolor="k",
                   s=72, lw=1.0, zorder=5,
                   label=f"labeled: {SPECIES_NAME[label]}")
    ax.scatter(*X_IRIS[next_query], s=300, facecolor="none", edgecolor="k",
               lw=2.6, zorder=6)
    ax.annotate("label this one next", xy=X_IRIS[next_query], xytext=(3.0, 2.52),
                fontsize="small", fontweight="bold", zorder=7,
                bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.3", lw=0.8),
                arrowprops=dict(arrowstyle="->", lw=1.6, color="k"))
    ax.legend(loc="lower right", fontsize="x-small", **LEGEND_STYLE)
    frame_panel(ax, "action", titled)

    return X_IRIS[next_query], pool_entropy.max()


IRIS_PANELS = {
    "prediction": panel_prediction,
    "generation": panel_generation,
    "discovery": panel_discovery,
    "action": panel_action,
}

In [ ]:
# One file per panel, untitled: the deck sets the question in its own
# typography and places the four panels itself.
# The larger base font is deliberate: these panels are shown four-up on one
# slide, so they are reduced much further than the Family A figures are.
panel_results = {}
for key, panel in IRIS_PANELS.items():
    with plt.rc_context({"font.size": 14}):
        fig, ax = plt.subplots(figsize=(7.2, 4.0))
        panel_results[key] = panel(ax, make_rng(f"iris_{key}"))
        fig.tight_layout()
        save_fig(fig, f"iris_{key}.svg")
        plt.show()

agreement = panel_results["discovery"]
next_flower, max_entropy = panel_results["action"]
print(f"\ndiscovery: the two inferred clusters agree with the species labels "
      f"for {agreement:.0%} of the 100 flowers (labels were not used to fit)")
print(f"action: next query at petal length {next_flower[0]} cm, "
      f"petal width {next_flower[1]} cm, predictive entropy {max_entropy:.3f} nats "
      f"(maximum possible is {np.log(2):.3f})")

In [ ]:
# A combined reference version, with in-figure titles. The slide deck does not
# use this file; it composes the four panels above. Kept for handouts.
fig, axes = plt.subplots(2, 2, figsize=(12.4, 9.6))
for ax, key in zip(axes.ravel(), ["prediction", "generation", "discovery", "action"]):
    IRIS_PANELS[key](ax, make_rng(f"iris_{key}"), titled=True)
fig.suptitle("Same flowers, same two measurements — four different questions",
             fontsize=15, fontweight="bold", y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.97))

save_fig(fig, "iris_four_questions.svg")
plt.show()

---

## Figure manifest

`slide_deck.typ` expects these files in `figures/`.

**Family A — different datasets for different tasks**

| File | Slide |
|---|---|
| `task_examples_figures__prediction_iris_logistic.svg` | Prediction: what should we say about a new case? |
| `task_examples_figures__generation_vae_samples.png` | Generation: what should we produce? |
| `task_examples_figures__discovery_gmm_clusters.svg` | Discovery: what structure explains the data? |
| `task_examples_figures__action_active_learning.svg` | Action: where should we measure next? |

**Family B — same dataset, different questions**

| File | Slide |
|---|---|
| `task_examples_figures__iris_prediction.svg` | Same flowers, four different questions (panel 1) |
| `task_examples_figures__iris_generation.svg` | Same flowers, four different questions (panel 2) |
| `task_examples_figures__iris_discovery.svg` | Same flowers, four different questions (panel 3) |
| `task_examples_figures__iris_action.svg` | Same flowers, four different questions (panel 4) |
| `task_examples_figures__iris_four_questions.svg` | Not used by the deck: combined reference version |

## Dependencies

On top of the NumPy/SciPy/Matplotlib baseline used by `coin_model.ipynb`, this
notebook needs `scikit-learn` (everywhere) and `torch` (cell A2 only). Both are
pinned in `pyproject.toml`. The Gaussian process in A4 is written directly in
NumPy/SciPy and adds no dependency.

If `torch` is unavailable, run every cell except A2's two cells: the
per-figure generators mean nothing else changes, and the committed
`generation_vae_samples.png` is left in place.